# Analyse SQL des données — Lerouge Moulin

Ce notebook permet d’interroger la base relationnelle SQLite créée par le pipeline ETL afin d’analyser les ventes, les clients, les paniers et les produits.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

In [2]:
PROJECT_DIR = Path.cwd().parent

DATABASE_PATH = (
    PROJECT_DIR
    / "database"
    / "lerouge_moulin.db"
)

print("Base trouvée :", DATABASE_PATH.exists())
print("Chemin :", DATABASE_PATH)

Base trouvée : True
Chemin : C:\Users\abide\projet_lerouge_moulin\database\lerouge_moulin.db


In [3]:
connexion = sqlite3.connect(DATABASE_PATH)

print("Connexion à la base réussie.")

Connexion à la base réussie.


In [4]:
requete_tables = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables_sql = pd.read_sql_query(
    requete_tables,
    connexion
)

display(tables_sql)

,name
0,CLIENT
1,LIGNE_PANIER
2,PANIER
3,PRODUIT


In [5]:
tables_attendues = [
    "CLIENT",
    "PRODUIT",
    "PANIER",
    "LIGNE_PANIER"
]

resultats = []

for table in tables_attendues:

    requete = f"""
    SELECT COUNT(*) AS nombre_lignes
    FROM {table};
    """

    nombre_lignes = pd.read_sql_query(
        requete,
        connexion
    ).iloc[0]["nombre_lignes"]

    resultats.append({
        "table": table,
        "nombre_lignes": nombre_lignes
    })

controle_tables = pd.DataFrame(resultats)

display(controle_tables)

,table,nombre_lignes
0,CLIENT,435
1,PRODUIT,200
2,PANIER,3333
3,LIGNE_PANIER,10343


In [6]:
for table in tables_attendues:

    print("=" * 50)
    print(table)
    print("=" * 50)

    requete = f"""
    SELECT *
    FROM {table}
    LIMIT 5;
    """

    display(
        pd.read_sql_query(
            requete,
            connexion
        )
    )

CLIENT


,client_id,name,age,sexe,opening_date,status
0,CLI0001,Julie Durand,81,M,2023-03-26,active
1,CLI0002,Laura Bernard,58,M,2022-10-30,active
2,CLI0003,Thomas Moreau,18,M,2025-03-07,active
3,CLI0004,Sophie Moreau,64,M,2025-08-03,active
4,CLI0005,Jean Petit,52,F,2021-06-20,active


PRODUIT


,product_id,product_name,product_price,price_date
0,PRD0001,Peinture 125,260.19,2026-07-08
1,PRD0002,Marteau 854,38.12,2026-07-24
2,PRD0003,Peinture 704,149.12,2026-07-01
3,PRD0004,Vis 338,178.11,2026-07-01
4,PRD0005,Vis 833,228.33,2026-07-18


PANIER


,cart_id,date,cart_price,client_id
0,CRT00519,2026-04-12,4507.13,CLI0349
1,CRT00992,2026-03-31,1606.03,CLI0335
2,CRT00421,2026-04-29,6573.80,CLI0246
3,CRT01364,2026-01-14,5839.25,CLI0249
4,CRT02032,2026-07-07,988.74,CLI0057


LIGNE_PANIER


,trsx_id,amount,price,products_price,cart_id,product_id
0,TRX000001,10,336.37,3363.70,CRT00519,PRD0152
1,TRX000002,5,164.79,823.95,CRT00992,PRD0050
2,TRX000003,8,288.19,2305.52,CRT00421,PRD0176
3,TRX000004,2,252.15,504.30,CRT01364,PRD0059
4,TRX000005,6,164.79,988.74,CRT02032,PRD0050


In [7]:
for table in tables_attendues:

    print("=" * 50)
    print("STRUCTURE DE", table)
    print("=" * 50)

    requete = f"""
    PRAGMA table_info({table});
    """

    structure = pd.read_sql_query(
        requete,
        connexion
    )

    display(structure)

STRUCTURE DE CLIENT


,cid,name,type,notnull,dflt_value,pk
0,0,client_id,TEXT,0,None,1
1,1,name,TEXT,1,None,0
2,2,age,INTEGER,1,None,0
3,3,sexe,TEXT,1,None,0
4,4,opening_date,TEXT,1,None,0
5,5,status,TEXT,1,None,0


STRUCTURE DE PRODUIT


,cid,name,type,notnull,dflt_value,pk
0,0,product_id,TEXT,0,None,1
1,1,product_name,TEXT,1,None,0
2,2,product_price,REAL,1,None,0
3,3,price_date,TEXT,1,None,0


STRUCTURE DE PANIER


,cid,name,type,notnull,dflt_value,pk
0,0,cart_id,TEXT,0,None,1
1,1,date,TEXT,1,None,0
2,2,cart_price,REAL,1,None,0
3,3,client_id,TEXT,1,None,0


STRUCTURE DE LIGNE_PANIER


,cid,name,type,notnull,dflt_value,pk
0,0,trsx_id,TEXT,0,None,1
1,1,amount,INTEGER,1,None,0
2,2,price,REAL,1,None,0
3,3,products_price,REAL,1,None,0
4,4,cart_id,TEXT,1,None,0
5,5,product_id,TEXT,1,None,0


## 1. Analyse de la période des ventes

Cette analyse permet d’identifier la première date de vente, la dernière date de vente et la durée totale couverte par les transactions.

In [8]:
requete_periode = """
SELECT
    MIN(date) AS premiere_date,
    MAX(date) AS derniere_date,
    COUNT(DISTINCT date) AS nombre_jours_avec_ventes
FROM PANIER;
"""

periode_ventes = pd.read_sql_query(
    requete_periode,
    connexion
)

display(periode_ventes)

,premiere_date,derniere_date,nombre_jours_avec_ventes
0,2026-01-01,2026-07-26,207


## 2. Chiffre d’affaires global

Le chiffre d’affaires est calculé à partir de la somme des montants des paniers.

In [10]:
requete_ca_global = """
SELECT
    ROUND(SUM(cart_price), 2) AS chiffre_affaires_total,
    COUNT(*) AS nombre_paniers,
    ROUND(AVG(cart_price), 2) AS panier_moyen,
    ROUND(MIN(cart_price), 2) AS panier_minimum,
    ROUND(MAX(cart_price), 2) AS panier_maximum
FROM PANIER;
"""

ca_global = pd.read_sql_query(
    requete_ca_global,
    connexion
)

display(ca_global)

,chiffre_affaires_total,nombre_paniers,panier_moyen,panier_minimum,panier_maximum
0,10575568.62,3333,3172.99,11.49,16103.4


## 3. Nombre de ventes par jour

In [19]:
requete_ventes_jour = """
SELECT
    date,
    COUNT(*) AS nombre_paniers,
    ROUND(SUM(cart_price), 2) AS chiffre_affaires
FROM PANIER
GROUP BY date
ORDER BY date;
"""

ventes_jour = pd.read_sql_query(
    requete_ventes_jour,
    connexion
)

display(ventes_jour.head(10))

,date,nombre_paniers,chiffre_affaires
0,2026-01-01,15,46200.30
1,2026-01-02,13,41544.27
2,2026-01-03,23,78961.12
3,2026-01-04,12,35727.87
4,2026-01-05,15,42435.07
5,2026-01-06,21,74841.07
6,2026-01-07,8,27954.92
7,2026-01-08,19,48031.02
8,2026-01-09,16,58199.01
9,2026-01-10,15,55402.19


### Interprétation

Les ventes sont enregistrées chaque jour sur la période étudiée. Le volume quotidien varie sensiblement d’une date à l’autre.

Sur les dix premiers jours de janvier, le nombre de paniers varie entre 8 et 23 par jour, tandis que le chiffre d’affaires quotidien se situe entre 27 954,92 € et 78 961,12 €.

Le 3 janvier 2026 se distingue dans cet extrait avec 23 paniers et un chiffre d’affaires de 78 961,12 €, alors que le 7 janvier présente l’activité la plus faible avec 8 paniers et 27 954,92 € de chiffre d’affaires.

Ces variations montrent que l’activité n’est pas parfaitement régulière au jour le jour.

## 4. Chiffre d’affaires cumulé

Une fonction de fenêtre SQL permet de calculer progressivement le chiffre d’affaires cumulé au fil du temps.

In [12]:
requete_ca_cumule = """
WITH ventes_journalieres AS (
    SELECT
        date,
        SUM(cart_price) AS chiffre_affaires_journalier
    FROM PANIER
    GROUP BY date
)

SELECT
    date,
    ROUND(chiffre_affaires_journalier, 2) AS chiffre_affaires_journalier,
    ROUND(
        SUM(chiffre_affaires_journalier)
        OVER (
            ORDER BY date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS chiffre_affaires_cumule
FROM ventes_journalieres
ORDER BY date;
"""

ca_cumule = pd.read_sql_query(
    requete_ca_cumule,
    connexion
)

display(ca_cumule.head(10))
display(ca_cumule.tail(10))

,date,chiffre_affaires_journalier,chiffre_affaires_cumule
0,2026-01-01,46200.30,46200.30
1,2026-01-02,41544.27,87744.57
2,2026-01-03,78961.12,166705.69
3,2026-01-04,35727.87,202433.56
4,2026-01-05,42435.07,244868.63
5,2026-01-06,74841.07,319709.70
6,2026-01-07,27954.92,347664.62
7,2026-01-08,48031.02,395695.64
8,2026-01-09,58199.01,453894.65
9,2026-01-10,55402.19,509296.84


,date,chiffre_affaires_journalier,chiffre_affaires_cumule
197,2026-07-17,45648.99,10127714.10
198,2026-07-18,34724.60,10162438.70
199,2026-07-19,62882.74,10225321.44
200,2026-07-20,61367.86,10286689.30
201,2026-07-21,62516.07,10349205.37
202,2026-07-22,40942.25,10390147.62
203,2026-07-23,57731.77,10447879.39
204,2026-07-24,43798.23,10491677.62
205,2026-07-25,33551.45,10525229.07
206,2026-07-26,50339.55,10575568.62


### Interprétation

Le chiffre d’affaires cumulé progresse de manière continue tout au long de la période, puisqu’il additionne successivement les ventes journalières.

Il passe de 46 200,30 € au 1er janvier 2026 à 509 296,84 € au 10 janvier 2026, pour atteindre finalement 10 575 568,62 € au 26 juillet 2026.

Cette mesure cumulative permet de suivre la progression globale de l’activité et de vérifier que le chiffre d’affaires total obtenu correspond bien au résultat calculé précédemment.

## 5. Tendance mensuelle des ventes

In [14]:
requete_ventes_mois = """
SELECT
    strftime('%Y-%m', date) AS mois,
    COUNT(*) AS nombre_paniers,
    ROUND(SUM(cart_price), 2) AS chiffre_affaires,
    ROUND(AVG(cart_price), 2) AS panier_moyen
FROM PANIER
GROUP BY strftime('%Y-%m', date)
ORDER BY mois;
"""

ventes_mois = pd.read_sql_query(
    requete_ventes_mois,
    connexion
)

display(ventes_mois)

,mois,nombre_paniers,chiffre_affaires,panier_moyen
0,2026-01,512,1649821.27,3222.31
1,2026-02,474,1503362.94,3171.65
2,2026-03,456,1426114.10,3127.44
3,2026-04,487,1596416.02,3278.06
4,2026-05,495,1585154.12,3202.33
5,2026-06,472,1415891.61,2999.77
6,2026-07,437,1398808.56,3200.93


### Interprétation

L’activité mensuelle reste globalement stable entre janvier et juillet 2026, avec un chiffre d’affaires mensuel compris entre environ 1,4 million d’euros et 1,65 million d’euros.

Le mois de janvier réalise le chiffre d’affaires le plus élevé avec 1 649 821,27 € et 512 paniers. À l’inverse, le mois de juin affiche le chiffre d’affaires le plus faible parmi les mois complets, avec 1 415 891,61 € et un panier moyen de 2 999,77 €.

Le mois d’avril présente le panier moyen le plus élevé, soit 3 278,06 €.

Le mois de juillet ne couvre que la période du 1er au 26 juillet. Son chiffre d’affaires de 1 398 808,56 € ne doit donc pas être directement comparé aux mois complets sans prendre en compte cette différence de durée.

Dans l’ensemble, aucune croissance ou baisse continue ne se dégage clairement. L’activité connaît plutôt des fluctuations modérées selon les mois.

## 6. Produits les plus vendus en quantité

In [15]:
requete_top_produits_quantite = """
SELECT
    p.product_id,
    p.product_name,
    SUM(lp.amount) AS quantite_vendue,
    COUNT(DISTINCT lp.cart_id) AS nombre_paniers,
    ROUND(SUM(lp.products_price), 2) AS chiffre_affaires
FROM LIGNE_PANIER AS lp
JOIN PRODUIT AS p
    ON lp.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name
ORDER BY quantite_vendue DESC
LIMIT 10;
"""

top_produits_quantite = pd.read_sql_query(
    requete_top_produits_quantite,
    connexion
)

display(top_produits_quantite)

,product_id,product_name,quantite_vendue,nombre_paniers,chiffre_affaires
0,PRD0121,Marteau 861,405,67,122257.35
1,PRD0178,Marteau 930,399,63,79037.91
2,PRD0172,Pince 229,395,67,38638.90
3,PRD0194,Ampoule 140,392,65,122135.44
4,PRD0049,Vis 196,376,61,13603.68
5,PRD0122,Vis 960,375,63,84033.75
6,PRD0132,Vis 379,373,59,57423.35
7,PRD0080,Marteau 752,372,64,127588.56
8,PRD0146,Ampoule 793,362,59,115130.48
9,PRD0188,Ampoule 777,362,62,102033.32


### Interprétation

Le produit le plus vendu en quantité est le Marteau 861, avec 405 unités réparties dans 67 paniers. Il génère un chiffre d’affaires de 122 257,35 €.

Il est suivi du Marteau 930 avec 399 unités, puis de la Pince 229 avec 395 unités.

Le classement montre que les produits les plus vendus en volume ne sont pas nécessairement ceux qui génèrent le plus de chiffre d’affaires. Par exemple, la Pince 229 occupe la troisième place en quantité, mais ne génère que 38 638,90 €, ce qui suggère un prix unitaire inférieur à celui de certains autres produits.

## 7. Produits générant le plus de chiffre d’affaires

In [16]:
requete_top_produits_ca = """
SELECT
    p.product_id,
    p.product_name,
    SUM(lp.amount) AS quantite_vendue,
    ROUND(SUM(lp.products_price), 2) AS chiffre_affaires
FROM LIGNE_PANIER AS lp
JOIN PRODUIT AS p
    ON lp.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name
ORDER BY chiffre_affaires DESC
LIMIT 10;
"""

top_produits_ca = pd.read_sql_query(
    requete_top_produits_ca,
    connexion
)

display(top_produits_ca)

,product_id,product_name,quantite_vendue,chiffre_affaires
0,PRD0080,Marteau 752,372,127588.56
1,PRD0121,Marteau 861,405,122257.35
2,PRD0194,Ampoule 140,392,122135.44
3,PRD0146,Ampoule 793,362,115130.48
4,PRD0112,Perceuse 350,327,112625.34
5,PRD0055,Perceuse 767,318,111220.50
6,PRD0119,Ampoule 424,324,107775.36
7,PRD0130,Tournevis 973,340,106199.00
8,PRD0060,Planche 322,313,106147.69
9,PRD0152,Peinture 543,307,103265.59


### Interprétation

Le produit générant le plus de chiffre d’affaires est le Marteau 752, avec 127 588,56 € pour 372 unités vendues.

Il est suivi du Marteau 861, avec 122 257,35 €, puis de l’Ampoule 140, avec 122 135,44 €.

Le Marteau 861 apparaît à la fois parmi les produits les plus vendus en quantité et parmi les produits générant le plus de chiffre d’affaires. Il constitue donc un produit particulièrement performant.

À l’inverse, certains produits peuvent avoir un volume de ventes élevé sans figurer en tête du chiffre d’affaires, en raison d’un prix unitaire plus faible.

## 8. Clients générant le plus de chiffre d’affaires

In [17]:
requete_top_clients = """
SELECT
    c.client_id,
    c.name,
    c.status,
    COUNT(p.cart_id) AS nombre_paniers,
    ROUND(SUM(p.cart_price), 2) AS chiffre_affaires,
    ROUND(AVG(p.cart_price), 2) AS panier_moyen
FROM CLIENT AS c
JOIN PANIER AS p
    ON c.client_id = p.client_id
GROUP BY
    c.client_id,
    c.name,
    c.status
ORDER BY chiffre_affaires DESC
LIMIT 10;
"""

top_clients = pd.read_sql_query(
    requete_top_clients,
    connexion
)

display(top_clients)

,client_id,name,status,nombre_paniers,chiffre_affaires,panier_moyen
0,CLI0003,Thomas Moreau,active,15,60344.36,4022.96
1,CLI0357,Jean Robert,active,13,58616.14,4508.93
2,CLI0064,Pierre Richard,active,16,54733.89,3420.87
3,CLI0243,Jean Richard,active,15,53071.23,3538.08
4,CLI0359,Marie Petit,active,12,51344.91,4278.74
5,CLI0167,Sophie Laurent,active,14,51189.15,3656.37
6,CLI0071,Jean Thomas,active,10,49784.08,4978.41
7,CLI0355,Luc Bernard,active,13,49176.51,3782.81
8,CLI0052,Pierre Thomas,active,11,48995.72,4454.16
9,CLI0198,Pierre Petit,active,11,48575.43,4415.95


### Interprétation

Thomas Moreau est le client ayant généré le chiffre d’affaires le plus élevé sur la période, avec 60 344,36 € répartis sur 15 paniers. Son panier moyen s’élève à 4 022,96 €.

Jean Robert arrive en deuxième position avec 58 616,14 € et un panier moyen de 4 508,93 €.

Pierre Richard occupe la troisième place avec 54 733,89 € et 16 paniers, soit le nombre de paniers le plus élevé parmi les trois premiers clients.

Le classement montre que le chiffre d’affaires généré par un client dépend à la fois de la fréquence de ses achats et du montant moyen de ses paniers.

## 9. Analyse des ventes par statut client

In [18]:
requete_statut_client = """
SELECT
    c.status,
    COUNT(DISTINCT c.client_id) AS nombre_clients,
    COUNT(p.cart_id) AS nombre_paniers,
    ROUND(SUM(p.cart_price), 2) AS chiffre_affaires,
    ROUND(AVG(p.cart_price), 2) AS panier_moyen
FROM CLIENT AS c
LEFT JOIN PANIER AS p
    ON c.client_id = p.client_id
GROUP BY c.status
ORDER BY chiffre_affaires DESC;
"""

analyse_statut = pd.read_sql_query(
    requete_statut_client,
    connexion
)

display(analyse_statut)

,status,nombre_clients,nombre_paniers,chiffre_affaires,panier_moyen
0,active,389,3067,9711785.69,3166.54
1,inactive,46,266,863782.93,3247.30


### Interprétation

Les clients actifs représentent la grande majorité de la clientèle, avec 389 clients sur 435, soit environ 89,4 % des clients.

Ils ont réalisé 3 067 paniers et généré 9 711 785,69 € de chiffre d’affaires, soit environ 91,8 % du chiffre d’affaires total.

Les 46 clients inactifs ont tout de même réalisé 266 paniers pour un chiffre d’affaires de 863 782,93 €.

Le panier moyen des clients inactifs, soit 3 247,30 €, est légèrement supérieur à celui des clients actifs, qui est de 3 166,54 €. Toutefois, leur poids global dans l’activité reste nettement inférieur en raison de leur faible nombre et de leur volume d’achat plus limité.

In [20]:
total_clients = analyse_statut["nombre_clients"].sum()
total_ca = analyse_statut["chiffre_affaires"].sum()

analyse_statut["part_clients_pct"] = (
    analyse_statut["nombre_clients"]
    / total_clients
    * 100
).round(2)

analyse_statut["part_ca_pct"] = (
    analyse_statut["chiffre_affaires"]
    / total_ca
    * 100
).round(2)

display(analyse_statut)

,status,nombre_clients,nombre_paniers,chiffre_affaires,panier_moyen,part_clients_pct,part_ca_pct
0,active,389,3067,9711785.69,3166.54,89.43,91.83
1,inactive,46,266,863782.93,3247.30,10.57,8.17


### Interprétation

Les clients actifs représentent 89,43 % de la clientèle et génèrent 91,83 % du chiffre d’affaires total. Ils constituent donc le principal moteur de l’activité commerciale.

Les clients inactifs ne représentent que 10,57 % des clients et 8,17 % du chiffre d’affaires.

Leur panier moyen reste néanmoins légèrement supérieur à celui des clients actifs, avec 3 247,30 € contre 3 166,54 €. Cela signifie que les clients inactifs achètent moins fréquemment, mais que leurs commandes peuvent rester relativement importantes.

La priorité commerciale semble donc être de fidéliser les clients actifs, tout en étudiant les raisons de l’inactivité des autres clients afin d’envisager des actions de réactivation.

# Analyse des données avec les DataFrames Python

Cette partie reprend les principales analyses réalisées en SQL en utilisant les fonctionnalités de la bibliothèque pandas.

In [33]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

client = pd.read_csv(
    PROCESSED_DIR / "client.csv",
    parse_dates=["opening_date"]
)

produit = pd.read_csv(
    PROCESSED_DIR / "produit.csv",
    parse_dates=["price_date"]
)

display(produit.head())

panier = pd.read_csv(
    PROCESSED_DIR / "panier.csv",
    parse_dates=["date"]
)

ligne_panier = pd.read_csv(
    PROCESSED_DIR / "ligne_panier.csv"
)

print("Chargement terminé.")

,product_id,product_name,product_price_original,currency_original,price_date,product_source
0,PRD0001,Peinture 125,260.19,EUR,2026-07-08,INTERNE
1,PRD0002,Marteau 854,38.12,EUR,2026-07-24,INTERNE
2,PRD0003,Peinture 704,149.12,EUR,2026-07-01,INTERNE
3,PRD0004,Vis 338,178.11,EUR,2026-07-01,INTERNE
4,PRD0005,Vis 833,228.33,EUR,2026-07-18,INTERNE


Chargement terminé.


In [34]:
dimensions = pd.DataFrame({
    "Table": [
        "CLIENT",
        "PRODUIT",
        "PANIER",
        "LIGNE_PANIER"
    ],
    "Nombre de lignes": [
        len(client),
        len(produit),
        len(panier),
        len(ligne_panier)
    ]
})

display(dimensions)

,Table,Nombre de lignes
0,CLIENT,435
1,PRODUIT,200
2,PANIER,3333
3,LIGNE_PANIER,10343


In [35]:
periode = pd.DataFrame({
    "Première date": [panier["date"].min()],
    "Dernière date": [panier["date"].max()],
    "Nombre de jours": [
        panier["date"].nunique()
    ]
})

display(periode)

,Première date,Dernière date,Nombre de jours
0,2026-01-01 00:27:51,2026-07-26 23:49:09,3333


In [36]:
ca = pd.DataFrame({
    "CA total": [
        panier["cart_price"].sum()
    ],
    "Nombre paniers": [
        len(panier)
    ],
    "Panier moyen": [
        panier["cart_price"].mean()
    ],
    "Panier min": [
        panier["cart_price"].min()
    ],
    "Panier max": [
        panier["cart_price"].max()
    ]
})

display(
    ca.round(2)
)

,CA total,Nombre paniers,Panier moyen,Panier min,Panier max
0,10575568.62,3333,3172.99,11.49,16103.4


In [37]:
ventes_jour = (
    panier
    .groupby("date")
    .agg(
        nombre_paniers=("cart_id", "count"),
        chiffre_affaires=("cart_price", "sum")
    )
    .reset_index()
)

display(
    ventes_jour.head(10)
)

,date,nombre_paniers,chiffre_affaires
0,2026-01-01 00:27:51,1,1991.23
1,2026-01-01 01:15:43,1,3529.62
2,2026-01-01 01:22:39,1,3542.64
3,2026-01-01 04:18:13,1,1330.56
4,2026-01-01 05:20:31,1,2657.83
5,2026-01-01 05:24:42,1,2551.25
6,2026-01-01 05:34:16,1,2345.46
7,2026-01-01 07:24:04,1,4161.92
8,2026-01-01 11:18:57,1,2054.89
9,2026-01-01 12:25:42,1,804.95


In [38]:
ventes_jour["ca_cumule"] = (
    ventes_jour["chiffre_affaires"]
    .cumsum()
)

display(
    ventes_jour.tail()
)

,date,nombre_paniers,chiffre_affaires,ca_cumule
3328,2026-07-26 18:51:36,1,2046.51,10560342.66
3329,2026-07-26 20:21:13,1,6569.53,10566912.19
3330,2026-07-26 20:58:54,1,4492.29,10571404.48
3331,2026-07-26 21:50:47,1,2604.96,10574009.44
3332,2026-07-26 23:49:09,1,1559.18,10575568.62


In [39]:
panier["mois"] = (
    panier["date"]
    .dt.to_period("M")
)

ventes_mois = (
    panier
    .groupby("mois")
    .agg(
        nombre_paniers=("cart_id", "count"),
        chiffre_affaires=("cart_price", "sum"),
        panier_moyen=("cart_price", "mean")
    )
    .reset_index()
)

display(
    ventes_mois.round(2)
)

,mois,nombre_paniers,chiffre_affaires,panier_moyen
0,2026-01,512,1649821.27,3222.31
1,2026-02,474,1503362.94,3171.65
2,2026-03,456,1426114.10,3127.44
3,2026-04,487,1596416.02,3278.06
4,2026-05,495,1585154.12,3202.33
5,2026-06,472,1415891.61,2999.77
6,2026-07,437,1398808.56,3200.93


In [40]:
top_produits = (
    ligne_panier
    .merge(
        produit,
        on="product_id"
    )
    .groupby(
        ["product_id",
        "product_name",
        "product_source",
        "currency_original"]
    )
    .agg(
        quantite_vendue=("amount", "sum"),
        nombre_paniers=("cart_id", "nunique"),
        chiffre_affaires=("products_price", "sum")
    )
    .sort_values(
        "quantite_vendue",
        ascending=False
    )
    .head(10)
    .reset_index()
)

display(
    top_produits.round(2)
)

,product_id,product_name,product_source,currency_original,quantite_vendue,nombre_paniers,chiffre_affaires
0,PRD0121,Marteau 861,INTERNE,EUR,405,67,122257.35
1,PRD0178,Marteau 930,INTERNE,EUR,399,63,79037.91
2,PRD0172,Pince 229,INTERNE,EUR,395,67,38638.90
3,PRD0194,Ampoule 140,EXTERNE,USD,392,65,122135.44
4,PRD0049,Vis 196,INTERNE,EUR,376,61,13603.68
5,PRD0122,Vis 960,INTERNE,EUR,375,63,84033.75
6,PRD0132,Vis 379,INTERNE,EUR,373,59,57423.35
7,PRD0080,Marteau 752,INTERNE,EUR,372,64,127588.56
8,PRD0188,Ampoule 777,EXTERNE,USD,362,62,102033.32
9,PRD0146,Ampoule 793,INTERNE,EUR,362,59,115130.48


In [29]:
top_clients = (
    panier
    .merge(
        client,
        on="client_id"
    )
    .groupby(
        ["client_id", "name", "status"]
    )
    .agg(
        nombre_paniers=("cart_id", "count"),
        chiffre_affaires=("cart_price", "sum"),
        panier_moyen=("cart_price", "mean")
    )
    .sort_values(
        "chiffre_affaires",
        ascending=False
    )
    .head(10)
    .reset_index()
)

display(
    top_clients.round(2)
)

,client_id,name,status,nombre_paniers,chiffre_affaires,panier_moyen
0,CLI0003,Thomas Moreau,active,15,60344.36,4022.96
1,CLI0357,Jean Robert,active,13,58616.14,4508.93
2,CLI0064,Pierre Richard,active,16,54733.89,3420.87
3,CLI0243,Jean Richard,active,15,53071.23,3538.08
4,CLI0359,Marie Petit,active,12,51344.91,4278.74
5,CLI0167,Sophie Laurent,active,14,51189.15,3656.37
6,CLI0071,Jean Thomas,active,10,49784.08,4978.41
7,CLI0355,Luc Bernard,active,13,49176.51,3782.81
8,CLI0052,Pierre Thomas,active,11,48995.72,4454.16
9,CLI0198,Pierre Petit,active,11,48575.43,4415.95


In [30]:
analyse_statut = (
    panier
    .merge(
        client,
        on="client_id"
    )
    .groupby("status")
    .agg(
        nombre_clients=("client_id", "nunique"),
        nombre_paniers=("cart_id", "count"),
        chiffre_affaires=("cart_price", "sum"),
        panier_moyen=("cart_price", "mean")
    )
    .reset_index()
)

analyse_statut["part_clients_pct"] = (
    analyse_statut["nombre_clients"]
    / analyse_statut["nombre_clients"].sum()
    * 100
)

analyse_statut["part_ca_pct"] = (
    analyse_statut["chiffre_affaires"]
    / analyse_statut["chiffre_affaires"].sum()
    * 100
)

display(
    analyse_statut.round(2)
)

,status,nombre_clients,nombre_paniers,chiffre_affaires,panier_moyen,part_clients_pct,part_ca_pct
0,active,389,3067,9711785.69,3166.54,89.43,91.83
1,inactive,46,266,863782.93,3247.30,10.57,8.17


### Analyse du catalogue produit après homogénéisation

#### 1. Analyse SQL : répartition par source

In [42]:
requete_source = """
SELECT
    product_source,
    COUNT(*) AS nombre_produits
FROM PRODUIT
GROUP BY product_source
ORDER BY nombre_produits DESC;
"""

analyse_source_sql = pd.read_sql_query(
    requete_source,
    connexion
)

display(analyse_source_sql)

,product_source,nombre_produits
0,INTERNE,180
1,EXTERNE,20


#### 2. Analyse SQL : répartition par devise

In [43]:
requete_devise = """
SELECT
    currency_original,
    COUNT(*) AS nombre_produits
FROM PRODUIT
GROUP BY currency_original
ORDER BY nombre_produits DESC;
"""

analyse_devise_sql = pd.read_sql_query(
    requete_devise,
    connexion
)

display(analyse_devise_sql)

,currency_original,nombre_produits
0,EUR,180
1,USD,20


#### 3. Analyse SQL : synthèse complète du catalogue

In [44]:
requete_catalogue = """
SELECT
    product_source,
    currency_original,
    COUNT(*) AS nombre_produits,
    ROUND(AVG(product_price_original), 2) AS prix_moyen,
    ROUND(MIN(product_price_original), 2) AS prix_minimum,
    ROUND(MAX(product_price_original), 2) AS prix_maximum
FROM PRODUIT
GROUP BY
    product_source,
    currency_original
ORDER BY product_source;
"""

analyse_catalogue_sql = pd.read_sql_query(
    requete_catalogue,
    connexion
)

display(analyse_catalogue_sql)

,product_source,currency_original,nombre_produits,prix_moyen,prix_minimum,prix_maximum
0,EXTERNE,USD,20,182.47,11.49,349.85
1,INTERNE,EUR,180,185.07,4.55,349.75


##### Analyses équivalentes avec Pandas
4. Répartition par source avec Pandas

In [45]:
analyse_source_pandas = (
    produit
    .groupby("product_source")
    .size()
    .reset_index(name="nombre_produits")
    .sort_values(
        by="nombre_produits",
        ascending=False
    )
)

display(analyse_source_pandas)

,product_source,nombre_produits
1,INTERNE,180
0,EXTERNE,20


#### 5. Répartition par devise avec Pandas

In [46]:
analyse_devise_pandas = (
    produit
    .groupby("currency_original")
    .size()
    .reset_index(name="nombre_produits")
    .sort_values(
        by="nombre_produits",
        ascending=False
    )
)

display(analyse_devise_pandas)

,currency_original,nombre_produits
0,EUR,180
1,USD,20


#### 6. Synthèse du catalogue avec Pandas

In [47]:
analyse_catalogue_pandas = (
    produit
    .groupby(
        [
            "product_source",
            "currency_original"
        ]
    )
    .agg(
        nombre_produits=("product_id", "count"),
        prix_moyen=("product_price_original", "mean"),
        prix_minimum=("product_price_original", "min"),
        prix_maximum=("product_price_original", "max")
    )
    .reset_index()
)

analyse_catalogue_pandas[
    [
        "prix_moyen",
        "prix_minimum",
        "prix_maximum"
    ]
] = analyse_catalogue_pandas[
    [
        "prix_moyen",
        "prix_minimum",
        "prix_maximum"
    ]
].round(2)

display(analyse_catalogue_pandas)

,product_source,currency_original,nombre_produits,prix_moyen,prix_minimum,prix_maximum
0,EXTERNE,USD,20,182.47,11.49,349.85
1,INTERNE,EUR,180,185.07,4.55,349.75


#### 7. Contrôle de cohérence source/devise

In [48]:
requete_coherence = """
SELECT
    product_id,
    product_name,
    product_source,
    currency_original
FROM PRODUIT
WHERE
    (
        product_source = 'INTERNE'
        AND currency_original <> 'EUR'
    )
    OR
    (
        product_source = 'EXTERNE'
        AND currency_original <> 'USD'
    );
"""

incoherences_sql = pd.read_sql_query(
    requete_coherence,
    connexion
)

display(incoherences_sql)

print(
    "Nombre d'incohérences source/devise :",
    len(incoherences_sql)
)

,product_id,product_name,product_source,currency_original


Nombre d'incohérences source/devise : 0


In [49]:
incoherences_pandas = produit[
    (
        (produit["product_source"] == "INTERNE")
        & (produit["currency_original"] != "EUR")
    )
    |
    (
        (produit["product_source"] == "EXTERNE")
        & (produit["currency_original"] != "USD")
    )
]

display(incoherences_pandas)

print(
    "Nombre d'incohérences source/devise :",
    len(incoherences_pandas)
)

,product_id,product_name,product_price_original,currency_original,price_date,product_source


Nombre d'incohérences source/devise : 0
